本 notebook 是 [M05.3 PyTorch 工程基础](../notes/03-pytorch-engineering.qmd) 的实例讲解。

1. 1. Tensor 的存储、视图与边界
2. 2. 参数、buffer 与 Dataset 契约
3. 3. 可恢复的手写训练循环
4. 4. 评估模式与输出审计

在 VS Code 中选择仓库 `.venv` 内核，逐单元运行；运行结果随文件保存。示例使用固定随机种子和合成数据。

运行前安装已声明依赖：`uv sync --locked --extra perf --extra data --extra dl`。本例使用 CPU，不需外部数据或预训练权重。

In [1]:
import copy
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import torch
from matplotlib_inline.backend_inline import set_matplotlib_formats
from torch import nn
from torch.utils.data import DataLoader, Dataset

from semiai.plotting import use_style

use_style()
set_matplotlib_formats("svg")
rng = np.random.default_rng(42)

## 1. Tensor 的存储、视图与边界

检查转置后的 stride 和连续性，再对比 detach、clone 与 NumPy 共享存储。

In [2]:
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
a = torch.arange(12, dtype=torch.float64).reshape(3, 4)
b = a.T
print("shape/stride:", a.shape, a.stride(), b.shape, b.stride(), "连续:", b.is_contiguous())
try:
    b.view(-1)
except RuntimeError as error:
    print("转置后 view:", type(error).__name__)
c = b.reshape(-1)
print("reshape 是否共享起始地址:", c.data_ptr() == a.data_ptr())
x = torch.tensor([1.0, 2.0], requires_grad=True)
detached = x.detach()
cloned = x.detach().clone()
with torch.no_grad():
    detached[0] = 9.0
print("detach 共享后的 x:", x, "clone:", cloned)
array = np.array([1.0, 2.0])
shared = torch.as_tensor(array)
copied = torch.tensor(array)
array[0] = 7.0
print("as_tensor/tensor:", shared, copied)
assert x[0] == 9 and cloned[0] == 1 and shared[0] == 7 and copied[0] == 1

shape/stride: torch.Size([3, 4]) (4, 1) torch.Size([4, 3]) (1, 4) 连续: False
转置后 view: RuntimeError
reshape 是否共享起始地址: False
detach 共享后的 x: tensor([9., 2.], requires_grad=True) clone: tensor([1., 2.])
as_tensor/tensor: tensor([7., 2.], dtype=torch.float64) tensor([1., 2.], dtype=torch.float64)


读法：detach 切断梯度但不复制存储，reshape 可能复制；不能仅根据形状推断别名关系。科学数据接口还需显式统一 dtype、单位与设备。

## 2. 参数、buffer 与 Dataset 契约

Module 自动注册子层和 Parameter，非训练状态用 buffer；数据集同时返回稳定 ID，方便审计 batch 顺序。

In [3]:
class Regressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("input_center", torch.zeros(2))
        self.layers = nn.Sequential(nn.Linear(2, 12), nn.Tanh(), nn.Dropout(0.15), nn.Linear(12, 1))

    def forward(self, x):
        return self.layers(x - self.input_center)


class SyntheticDataset(Dataset):
    def __init__(self):
        generator = torch.Generator().manual_seed(7)
        self.x = torch.randn(131, 2, generator=generator)
        self.y = (2 * self.x[:, 0] - self.x[:, 1] + 0.1 * torch.randn(131, generator=generator))[
            :, None
        ]

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], self.y[index], index


dataset = SyntheticDataset()
model = Regressor()
print("参数名称:", [name for name, _ in model.named_parameters()])
print("buffer:", list(model.named_buffers()), "state_dict keys:", list(model.state_dict()))
loader = DataLoader(
    dataset, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(9), num_workers=0
)
print("batch 大小:", [len(batch[0]) for batch in loader])
assert "input_center" in model.state_dict() and "input_center" not in dict(model.named_parameters())

参数名称: ['layers.0.weight', 'layers.0.bias', 'layers.3.weight', 'layers.3.bias']
buffer: [('input_center', tensor([0., 0.]))] state_dict keys: ['input_center', 'layers.0.weight', 'layers.0.bias', 'layers.3.weight', 'layers.3.bias']
batch 大小: [32, 32, 32, 32, 3]


读法：最后一个 batch 只有三行，epoch 平均损失应按样本数加权；普通 Python list 中的子层不会像 ModuleList 那样自动注册。示例 num_workers=0 避免把并行采样细节混入恢复核验。

## 3. 可恢复的手写训练循环

比较连续八轮与三轮后保存再恢复五轮，恢复模型、优化器、调度器、全局随机状态和数据加载器随机状态。

In [4]:
def build():
    torch.manual_seed(42)
    model = Regressor()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
    generator = torch.Generator().manual_seed(9)
    loader = DataLoader(dataset, batch_size=32, shuffle=True, generator=generator, num_workers=0)
    return model, optimizer, scheduler, generator, loader


def train_epochs(model, optimizer, scheduler, loader, count):
    history = []
    orders = []
    for _ in range(count):
        model.train()
        total = 0.0
        samples = 0
        order = []
        for x, y, ids in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.mse_loss(model(x), y)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(x)
            samples += len(x)
            order.extend(ids.tolist())
        scheduler.step()
        history.append(total / samples)
        orders.append(order)
    return history, orders


full, optimizer, scheduler, generator, loader = build()
full_history, full_order = train_epochs(full, optimizer, scheduler, loader, 8)
full_state = copy.deepcopy(full.state_dict())
partial, optimizer, scheduler, generator, loader = build()
first_history, first_order = train_epochs(partial, optimizer, scheduler, loader, 3)
checkpoint = {
    "epoch": 3,
    "model": partial.state_dict(),
    "optimizer": optimizer.state_dict(),
    "scheduler": scheduler.state_dict(),
    "torch_rng": torch.get_rng_state(),
    "loader_rng": generator.get_state(),
}
with TemporaryDirectory() as directory:
    path = Path(directory) / "checkpoint.pt"
    torch.save(checkpoint, path)
    loaded = torch.load(path, map_location="cpu", weights_only=True)
    resumed, optimizer, scheduler, generator, loader = build()
    resumed.load_state_dict(loaded["model"])
    optimizer.load_state_dict(loaded["optimizer"])
    scheduler.load_state_dict(loaded["scheduler"])
    generator.set_state(loaded["loader_rng"])
    torch.set_rng_state(loaded["torch_rng"])
    later_history, later_order = train_epochs(
        resumed, optimizer, scheduler, loader, 8 - loaded["epoch"]
    )
resumed_history = first_history + later_history
max_difference = max(
    (full_state[k] - resumed.state_dict()[k]).abs().max().item() for k in full_state
)
print("连续训练 loss:", full_history, "\n恢复训练 loss:", resumed_history)
print("参数最大差:", max_difference, "batch 顺序相同:", full_order == first_order + later_order)
assert max_difference == 0 and full_history == resumed_history
assert full_order == first_order + later_order

连续训练 loss: [3.9108631519870904, 2.1860434299206917, 1.0330814969448643, 0.4636781223630177, 0.3095822809761717, 0.28739410493778816, 0.37648827520035605, 0.3142997030993454] 
恢复训练 loss: [3.9108631519870904, 2.1860434299206917, 1.0330814969448643, 0.4636781223630177, 0.3095822809761717, 0.28739410493778816, 0.37648827520035605, 0.3142997030993454]
参数最大差: 0.0 batch 顺序相同: True


读法：核验范围是同一 CPU、版本、单进程 DataLoader、epoch 边界恢复；中途 batch、worker 预取、CUDA 和分布式状态需要另外的恢复协议。只存模型权重无法保证相同后续更新。

## 4. 评估模式与输出审计

切换 eval 并禁用梯度记录，按样本加权汇总误差；打印实际版本与模式，检查重复推理一致。

In [5]:
resumed.eval()
evaluation = DataLoader(dataset, batch_size=40, shuffle=False)
outputs = []
targets = []
with torch.inference_mode():
    for x, y, _ in evaluation:
        outputs.append(resumed(x))
        targets.append(y)
prediction = torch.cat(outputs)
target = torch.cat(targets)
print("固定训练数据上的 eval MSE（仅机制检查）:", nn.functional.mse_loss(prediction, target).item())
with torch.inference_mode():
    a = resumed(dataset.x)
    b = resumed(dataset.x)
print(
    "重复 eval 完全相同:",
    torch.equal(a, b),
    "torch version:",
    str(torch.__version__),
    "device:",
    next(resumed.parameters()).device,
)
assert torch.equal(a, b)
resumed.train()
with torch.no_grad():
    a = resumed(dataset.x)
    b = resumed(dataset.x)
print("train + no_grad 仍有 Dropout 随机性:", not torch.equal(a, b))

固定训练数据上的 eval MSE（仅机制检查）: 0.17916259169578552
重复 eval 完全相同: True torch version: 2.15.0 device: cpu
train + no_grad 仍有 Dropout 随机性: True


读法：eval 改模块行为，no_grad/inference_mode 控制图记录，两者分工不同。这里在训练数据上评价只核验模式与汇总，不能当作泛化成绩。

## 小结

训练工程首先是状态工程。形状与布局定义数据接口，注册与模式定义模型行为，优化器和随机源定义后续更新。明确恢复边界并实际比较连续训练与恢复训练，是比只写出保存命令更有力的核验。